Environment setup

In [14]:
import os
import sys
import platform
import subprocess
import shutil
import torch
import torchaudio
from data.tokenizer import AudioTokenizer, TextTokenizer
import json


In [7]:
IS_KAGGLE = os.path.exists('/kaggle/working')
IS_COLAB = 'google.colab' in str(get_ipython())

if IS_KAGGLE:
    print("Running on Kaggle - Internet must be ON")
elif IS_COLAB:
    print("Running on Colab")
else:
    print("Running locally on Windows")

Running locally on Windows


Colab ONLY: This will crash the kernel the first time. This is expected - if you rerun from the start, it will work.

In [8]:
if IS_COLAB:
    print("Running on Colab - Setting up Cloud environment...")
    # Colab only
    !apt-get install -y git-core ffmpeg espeak-ng libsox-dev
    !pip install -q condacolab
    import condacolab
    condacolab.install()
    condacolab.check()
else:
    # !apt-get install -y git-core ffmpeg espeak-ng libsox-dev
    print("Running locally \ Kaggle - using existing Conda environment.")

Running locally \ Kaggle - using existing Conda environment.


Clone repo

In [9]:
# Check if we are already inside the folder or if it's nearby
# Check if folder exists or if we are on Cloud (Kaggle/Colab)
if os.path.basename(os.getcwd()) == 'VoiceCraft':
    print("Already inside VoiceCraft folder.")
elif os.path.exists('VoiceCraft'):
    print("VoiceCraft folder found, entering...")
    # Always enter the project directory
    %cd VoiceCraft
else:
    print("Cloning repository for the first time...")
    !git clone https://github.com/jasonppy/VoiceCraft.git
    # Always enter the project directory
    %cd VoiceCraft

# Adding paths so Python can find the 'models' and 'src' folders
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())
    
src_path = os.path.join(os.getcwd(), "src")
if src_path not in sys.path:
    sys.path.append(src_path)

print(f"Current working directory: {os.getcwd()}")

Already inside VoiceCraft folder.
Current working directory: /home/sukiennik/projects/VoiceCraft


In [14]:
# --- Setup Cell for Kaggle/WSL ---
# This cell installs all necessary dependencies for VoiceCraft

# 1. Install system libraries (Specific for Linux/Kaggle)
# On Kaggle, we don't need 'sudo'
if os.path.exists('/kaggle'):
    !apt-get update && apt-get install -y ffmpeg espeak-ng libsox-dev
else:
    # For WSL, we usually need sudo
    print("Running on WSL, make sure to install ffmpeg and espeak-ng manually if this fails.")

# 2. Install specific versions to avoid the conflicts we saw earlier
!pip install setuptools==69.5.1
!pip install torchaudio==2.1.2 torch==2.1.2 --index-url https://download.pytorch.org/whl/cu118
!pip install xformers==0.0.23.post1 --index-url https://download.pytorch.org/whl/cu118

# 3. Install VoiceCraft dependencies
!pip install phonemizer==3.2.1 datasets==2.16.0 torchmetrics==0.11.1 huggingface_hub==0.22.2 tensorboard==2.16.2

# 4. Install Audiocraft (The engine behind VoiceCraft)
!pip install -e git+https://github.com/facebookresearch/audiocraft.git@c5157b5bf14bf83449c17ea1eeb66c19fb4bc7f0#egg=audiocraft

# 5. Fix for Phonemizer library path (Crucial for VoiceCraft)
os.environ["PHONEMIZER_ESPEAK_LIBRARY"] = "/usr/lib/x86_64-linux-gnu/libespeak-ng.so"
os.environ["PHONEMIZER_ESPEAK_PATH"] = "/usr/bin/espeak-ng"

print("✅ All dependencies installed and environment configured!")

Running on WSL, make sure to install ffmpeg and espeak-ng manually if this fails.
Looking in indexes: https://download.pytorch.org/whl/cu118
Looking in indexes: https://download.pytorch.org/whl/cu118
  Using cached huggingface_hub-0.22.2-py3-none-any.whl.metadata (12 kB)
Using cached huggingface_hub-0.22.2-py3-none-any.whl (388 kB)
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.2.0 requires huggingface-hub<2.0,>=1.3.0, but you have huggingface-hub 0.22.2 which is incompatible.
Obtaining audiocraft from git+https://github.com/facebookresearch/audiocraft.git@c5157b5bf14bf83449c17ea1eeb66c19fb4bc7f0#egg=audiocraft
  Skipping because already up-to-date.
  Installin

Windows only (for local running) : install relvance tourch for GPU 

In [10]:
IS_CLOUD = os.path.exists('/kaggle/working') or 'google.colab' in str(get_ipython()) # Local or cloud?

# Checking if Torch is already installed and functional

torch_installed = False
try:
    import torch
    if torch.cuda.is_available():
        torch_installed = True
        print(f"✅ Torch {torch.__version__} with CUDA is ready.")
except:
    print("❌ Torch is not installed or not working correctly.")

if not torch_installed:
    if not IS_CLOUD:
        print("Installing Torch for Windows (CUDA 11.8)...")
        !pip install torch==2.1.2+cu118 torchaudio==2.1.2+cu118 --index-url https://download.pytorch.org/whl/cu118
        import torch # reload after installtion 
    else:
        print("Cloud environment: Installing default Torch...")
        !pip install torch torchaudio
        import torch
else:
    print("Skipping Torch installation as it is already functional.")

✅ Torch 2.1.2+cu118 with CUDA is ready.
Skipping Torch installation as it is already functional.


In [16]:
# Main installation block
!conda install -y -c conda-forge montreal-forced-aligner=2.2.17 openfst=1.8.2 kaldi=5.5.1068
!pip install joblib==1.3.2 numpy==1.26.4 tensorboard datasets==2.16.0 torchmetrics==0.11.1 phonemizer==3.2.1
!pip install transformers==4.38.2 huggingface_hub==0.22.2

2 channel Terms of Service accepted
Channels:
 - conda-forge
 - defaults
Platform: linux-64
Solving environment: done


==> WARNING: A newer version of conda exists. <==
    current version: 25.11.1
    latest version: 26.1.1

Please update conda by running

    $ conda update -n base -c defaults conda



# All requested packages already installed.

  Using cached joblib-1.3.2-py3-none-any.whl.metadata (5.4 kB)
Using cached joblib-1.3.2-py3-none-any.whl (302 kB)
  Attempting uninstall: joblib
    Found existing installation: joblib 1.5.3
    Uninstalling joblib-1.5.3:
      Successfully uninstalled joblib-1.5.3
  Using cached transformers-4.38.2-py3-none-any.whl.metadata (130 kB)
  Using cached huggingface_hub-0.22.2-py3-none-any.whl.metadata (12 kB)
  Using cached tokenizers-0.15.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.7 kB)
Using cached transformers-4.38.2-py3-none-any.whl (8.5 MB)
Using cached huggingface_hub-0.22.2-py3-none-any.whl (388 kB)
Using ca

In [11]:
!mfa version

2.2.17


In [12]:
# Checking if the specific Meta AudioCraft version is already there
try:
    import audiocraft
except ImportError:
    import sys
    import os
    audiocraft_path = os.path.join(os.getcwd(), "src", "audiocraft")
    if os.path.exists(audiocraft_path):
        sys.path.append(audiocraft_path)
    
    # Try again
    try:
        import audiocraft
    except ImportError:
        # Installing Meta's Audiocraft
        print("❌ AudioCraft not found. Installing Meta's specific version...")
        !pip install -e git+https://github.com/facebookresearch/audiocraft.git@c5157b5bf14bf83449c17ea1eeb66c19fb4bc7f0#egg=audiocraft

import audiocraft
print(f"✅ AudioCraft is ready! (Location: {audiocraft.__file__})")


✅ AudioCraft is ready! (Location: /home/sukiennik/miniconda3/envs/voicecraft_linux/lib/python3.10/site-packages/audiocraft/__init__.py)


In [13]:
import sys
import importlib
import site

# English comment: Reload site-packages to ensure new installs are recognized
importlib.reload(site)

try:
    import audiocraft
    # English comment: Print the location to verify it's not 'None'
    print(f"✅ Found it! Location: {audiocraft.__file__}")
except (ImportError, NameError):
    print("⚠️ Still struggling. Forcing the path...")
    # English comment: Add the specific conda environment path to sys.path
    env_path = "/home/sukiennik/miniconda3/envs/voicecraft_linux/lib/python3.10/site-packages"
    if env_path not in sys.path:
        sys.path.insert(0, env_path)
    
    import audiocraft
    print(f"✅ Success after manual path injection! Location: {audiocraft.__file__}")

✅ Found it! Location: /home/sukiennik/miniconda3/envs/voicecraft_linux/lib/python3.10/site-packages/audiocraft/__init__.py


In [14]:
try:
    # Checking if VoiceCraft models can be imported
    import models.voicecraft as voicecraft
    print("✅ VoiceCraft models found and ready!")
except ImportError as e:
    print(f"❌ Could not find VoiceCraft models: {e}")
    print("Tip: Check if you are in the correct directory.")

try:
    import audiocraft
    print(f"✅ AudioCraft is ready! (Version: {audiocraft.__version__})")
except ImportError:
    print("⚠️ AudioCraft not found in environment. Attempting to link from src...")
    if os.path.exists("src/audiocraft"):
        sys.path.append(os.path.abspath("src/audiocraft"))
        import audiocraft
        print(f"✅ AudioCraft linked from src! (Version: {audiocraft.__version__})")
    else:
        !pip install -e src/audiocraft
        import audiocraft
        print(audiocraft.__version__)

✅ VoiceCraft models found and ready!
✅ AudioCraft is ready! (Version: 1.0.0)


In [ ]:
# Installing the latest compatible version for CUDA 11.8 from the provided list
# !pip install --force-reinstall xformers==0.0.27.post2+cu118 --index-url https://download.pytorch.org/whl/cu118

# Installing synchronized versions for Windows + CUDA 11.8
# !pip install torch==2.1.2+cu118 torchaudio==2.1.2+cu118 xformers==0.0.23.post1 --index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://download.pytorch.org/whl/cu118
     ---------------------------------------- 0.0/10.8 MB ? eta -:--:--
     ------ --------------------------------- 1.8/10.8 MB 9.1 MB/s eta 0:00:01
     --------------- ------------------------ 4.2/10.8 MB 10.5 MB/s eta 0:00:01
     ------------------------ --------------- 6.6/10.8 MB 10.6 MB/s eta 0:00:01
     --------------------------------- ------ 8.9/10.8 MB 10.9 MB/s eta 0:00:01
     ---------------------------------------- 10.8/10.8 MB 10.8 MB/s  0:00:00
  Using cached numpy-2.2.6-cp310-cp310-win_amd64.whl.metadata (60 kB)
     ---------------------------------------- 0.0/2.7 GB ? eta -:--:--
     ---------------------------------------- 0.0/2.7 GB 11.2 MB/s eta 0:04:02
     ---------------------------------------- 0.0/2.7 GB 11.4 MB/s eta 0:03:56
     ---------------------------------------- 0.0/2.7 GB 11.5 MB/s eta 0:03:55
     ---------------------------------------- 0.0/2.7 GB 11.5 MB/s eta 0:03:54
     ----------

  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 2.16.0 requires fsspec[http]<=2023.10.0,>=2023.1.0, but you have fsspec 2025.12.0 which is incompatible.
gradio 3.50.2 requires numpy~=1.0, but you have numpy 2.2.6 which is incompatible.
torchaudio 2.7.1+cu118 requires torch==2.7.1+cu118, but you have torch 2.4.0+cu118 which is incompatible.


In [ ]:
# Installing the Intel runtime libraries that provide missing DLLs like fbgemm
# !pip install mkl mkl-include

   ---------------------------------------- 0.0/155.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/155.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/155.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/155.5 MB ? eta -:--:--
   ---------------------------------------- 0.3/155.5 MB ? eta -:--:--
   ---------------------------------------- 0.5/155.5 MB 1.1 MB/s eta 0:02:19
   ---------------------------------------- 0.8/155.5 MB 1.6 MB/s eta 0:01:38
   ---------------------------------------- 1.8/155.5 MB 2.5 MB/s eta 0:01:02
    --------------------------------------- 2.1/155.5 MB 2.0 MB/s eta 0:01:16
    --------------------------------------- 2.9/155.5 MB 2.3 MB/s eta 0:01:07
    --------------------------------------- 3.4/155.5 MB 2.4 MB/s eta 0:01:03
   - -------------------------------------- 4.2/155.5 MB 2.6 MB/s eta 0:00:59
   - -------------------------------------- 5.0/155.5 MB 2.8 MB/s eta 0:00:55
   - ----------------

In [15]:
print(f"Torch version: {torch.__version__}")
print(f"Is CUDA available? {torch.cuda.is_available()}")
print("If you see True, the GPU is ready!")

Torch version: 2.1.2+cu118
Is CUDA available? True
If you see True, the GPU is ready!


In [16]:
import xformers
import audiocraft
print("Success! Everything is connected.")

Success! Everything is connected.


In [17]:
!mfa model download dictionary english_us_arpa && \
mfa model download acoustic english_us_arpa

 WARNING  Local version of model already exists                                 
          (/home/sukiennik/Documents/MFA/pretrained_models/dictionary/english_us
          _arpa.dict). Use the --ignore_cache flag to force redownloading.      
 WARNING  Local version of model already exists                                 
          (/home/sukiennik/Documents/MFA/pretrained_models/acoustic/english_us_a
          rpa.zip). Use the --ignore_cache flag to force redownloading.         


In [18]:
print("🚀 Environment is set and ready.")

🚀 Environment is set and ready.


Hebrew setup

In [ ]:
# Install the necessary library if not already installed
!pip install huggingface_hub datasets
!pip install ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 914.9/914.9 kB 7.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 11.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ipywidgets]


In [ ]:
# Log in to Hugging Face (you only need to do this once)
# This will prompt you for your Token from huggingface.co/settings/tokens
from huggingface_hub import notebook_login
notebook_login()

In [7]:
# import os
# from datasets import load_dataset

# # Define local path in WSL
# # local_data_path = "./hebrew_cv_data"
# local_data_path = "./fleurs_hebrew"

# # Download and load the Hebrew dataset
# # This will save the files to the local_data_path
# print("Starting download... this might take a while.")
# # print("version 17.0...")
# # dataset = load_dataset("mozilla-foundation/common_voice_17_0", "he", 
# #                         cache_dir=local_data_path, 
# #                         split="train") 

# print("Downloading Google FLEURS (Hebrew)...")
# # dataset = load_dataset("google/fleurs", "he_il", split="train", cache_dir=local_data_path, trust_remote_code=True)
# dataset = load_dataset(
#         "google/fleurs", 
#         "he_il", 
#         split="train", 
#         cache_dir=local_data_path,
#         trust_remote_code=True,
#         download_mode="force_redownload" 
#     )

# print(f"Dataset downloaded and saved to: {os.path.abspath(local_data_path)}")

In [5]:
import os
from datasets import load_dataset

# English comments: Load the already downloaded files from default cache
print("Loading the downloaded dataset...")

try:
    # We remove the cache_dir to let it find the files where it just saved them
    dataset = load_dataset(
        "google/fleurs", 
        "he_il", 
        split="train", 
        trust_remote_code=True
    )
    
    print("\n--- Success! ---")
    print(f"Dataset loaded. Number of samples: {len(dataset)}")
    
    # Check one sample to make sure audio and text are linked
    sample = dataset[0]
    print(f"Text: {sample['transcription']}")
    print(f"Audio array shape: {sample['audio']['array'].shape}")

except Exception as e:
    print(f"Error loading: {e}")
    print("If it fails, we will try to point it directly to the download folder.")

Loading the downloaded dataset...


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]


--- Success! ---
Dataset loaded. Number of samples: 3242
Text: זו הרכישה הגדולה ביותר בתולדות ebay
Audio array shape: (60480,)


In [6]:
# Save the loaded dataset to a local, visible directory
local_storage_path = os.path.abspath("./fleurs_hebrew_final")

print(f"Saving dataset to: {local_storage_path}...")
dataset.save_to_disk(local_storage_path)
print("Done! The dataset is now safely stored in your project folder.")

Saving dataset to: /home/sukiennik/projects/VoiceCraft/fleurs_hebrew_final...


Saving the dataset (0/5 shards):   0%|          | 0/3242 [00:00<?, ? examples/s]

Done! The dataset is now safely stored in your project folder.


In [1]:
# Validation download
import os
from datasets import load_dataset

# English comments: Load the already downloaded files from default cache
print("Loading the downloaded dataset...")

try:
    # We remove the cache_dir to let it find the files where it just saved them
    dataset_val = load_dataset(
        "google/fleurs", 
        "he_il", 
        split="validation", 
        trust_remote_code=True
    )
    
    print("\n--- Success! ---")
    print(f"Dataset loaded. Number of samples: {len(dataset_val)}")
    
    # Check one sample to make sure audio and text are linked
    sample = dataset_val[0]
    print(f"Text: {sample['transcription']}")
    print(f"Audio array shape: {sample['audio']['array'].shape}")

except Exception as e:
    print(f"Error loading: {e}")
    print("If it fails, we will try to point it directly to the download folder.")

Loading the downloaded dataset...


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]


--- Success! ---
Dataset loaded. Number of samples: 328
Text: כעת זמין באופן נרחב בכל רחבי הארכיפלג המטבח הג'אווני מאופיין במגוון מנות מתובלות בפשטות והטעמים הבולטים שהג'אוונסים אוהבים הם בוטנים פלפל צ'ילי סוכר בעיקר סוכר קוקוס ג'אווני ומגוון תבלינים ארומטיים
Audio array shape: (257280,)


In [3]:
print("Accessing Test split...")
dataset_test = load_dataset(
    "google/fleurs", 
    "he_il", 
    split="test", 
    trust_remote_code=True
)

print("Accessing Train split...")
dataset_train = load_dataset(
    "google/fleurs", 
    "he_il", 
    split="train", 
    trust_remote_code=True
)

Accessing Test split...


Accessing Train split...


In [2]:
# Save the loaded dataset to a local, visible directory
local_storage_path = os.path.abspath("./fleurs_hebrew/val")

print(f"Saving dataset to: {local_storage_path}...")
dataset_val.save_to_disk(local_storage_path)
print("Done! The dataset is now safely stored in your project folder.")

Saving dataset to: /home/sukiennik/projects/VoiceCraft/fleurs_hebrew/val...


Saving the dataset (0/1 shards):   0%|          | 0/328 [00:00<?, ? examples/s]

Done! The dataset is now safely stored in your project folder.


In [4]:
# Save the loaded dataset to a local, visible directory
local_storage_path = os.path.abspath("./fleurs_hebrew/train")

print(f"Saving dataset to: {local_storage_path}...")
dataset_train.save_to_disk(local_storage_path)
print("Done! The train dataset is now safely stored in your project folder.")

Saving dataset to: /home/sukiennik/projects/VoiceCraft/fleurs_hebrew/train...


Saving the dataset (0/5 shards):   0%|          | 0/3242 [00:00<?, ? examples/s]

Done! The train dataset is now safely stored in your project folder.


In [5]:
# Save the loaded dataset to a local, visible directory
local_storage_path = os.path.abspath("./fleurs_hebrew/test")

print(f"Saving dataset to: {local_storage_path}...")
dataset_test.save_to_disk(local_storage_path)
print("Done! The test dataset is now safely stored in your project folder.")

Saving dataset to: /home/sukiennik/projects/VoiceCraft/fleurs_hebrew/test...


Saving the dataset (0/1 shards):   0%|          | 0/792 [00:00<?, ? examples/s]

Done! The test dataset is now safely stored in your project folder.


In [12]:
text_tokenizer = TextTokenizer(backend="espeak")
# Re-initialize the tokenizer with Hebrew support
text_tokenizer_he = TextTokenizer(backend="espeak", language="he")

In [ ]:
# --- Main Manifest Creation Loop ---
base_dir = "./voicecraft_data/manifest"
os.makedirs(base_dir, exist_ok=True)
manifest_path = os.path.join(base_dir, "hebrew_manifest_val.jsonl")

# Directory for saving extracted WAV files
output_wav_dir = "./fleurs_hebrew/voicecraft_samples/val"
os.makedirs(output_wav_dir, exist_ok=True)

num_samples_val = len(dataset_val)
manifest = []

print(f"Starting to prepare manifest for {num_samples} samples...")

for i in range(num_samples):
    try:
        sample = dataset_val[i]
        raw_text = sample['transcription']

        # Save the audio file as WAV
        # Extracting audio array and sampling rate from the dataset
        audio_array = torch.tensor(sample['audio']['array']).unsqueeze(0)
        sr = sample['audio']['sampling_rate']

        # Calculate duration in Encodec tokens (VoiceCraft standard)
        # Encodec at 16kHz produces 50 tokens per second.
        # Formula: (num_samples / sampling_rate) * 50
        num_audio_samples = audio_array.shape[1]
        duration_frames = int((num_audio_samples / sr) * 50)
        
        wav_filename = f"sample_{i}.wav"
        wav_path = os.path.join(output_wav_dir, wav_filename)
        
        # Physical save to disk using torchaudio
        torchaudio.save(wav_path, audio_array, sr)
    
        # Get Phonemes
        phonemes_list = text_tokenizer_he(raw_text)[0]
        phonemes_str = " ".join(phonemes_list)
        
        # Append to manifest with relative path for VoiceCraft
        manifest.append({
            "audio_filepath": f"{output_wav_dir}/{wav_filename}",
            "text": raw_text,
            "phonemes": phonemes_str,
            "duration_frames": duration_frames
        })

        # Print sample 0 to verify everything looks correct
        if i == 0:
            print(f"--- DEBUG Sample 0 ---")
            print(f"Text: {raw_text}")
            print(f"Phonemes: {phonemes_str[:50]}...")
            print(f"Duration: {duration_frames} frames")
            print(f"----------------------")
        
        if i % 100 == 0: print(f"Processed {i} samples...")
            
    except Exception as e:
        print(f"Error in sample {i}: {e}")

# Save the manifest
with open(manifest_path, "w", encoding="utf-8") as f:
    for entry in manifest:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print("Done! Check 'hebrew_manifest_val.jsonl'")

Starting to prepare manifest for 328 samples...
--- DEBUG Sample 0 ---
Text: כעת זמין באופן נרחב בכל רחבי הארכיפלג המטבח הג'אווני מאופיין במגוון מנות מתובלות בפשטות והטעמים הבולטים שהג'אוונסים אוהבים הם בוטנים פלפל צ'ילי סוכר בעיקר סוכר קוקוס ג'אווני ומגוון תבלינים ארומטיים
Phonemes: χ ʔ t _ z m i n _ v ʔ v f n _ n ʁ χ v _ v χ l _ ʁ ...
Duration: 803 frames
----------------------
Processed 0 samples...
Processed 100 samples...
Processed 200 samples...
Processed 300 samples...
Done! Check 'hebrew_manifest_val.jsonl'


In [ ]:
# --- Main Manifest Creation Loop ---
base_dir = "./voicecraft_data/manifest"
os.makedirs(base_dir, exist_ok=True)
manifest_path = os.path.join(base_dir, "hebrew_manifest_test.jsonl")

# Directory for saving extracted WAV files
output_wav_dir = "./fleurs_hebrew/voicecraft_samples/test"
os.makedirs(output_wav_dir, exist_ok=True)

num_samples_train_orig = len(dataset_train)
num_samples_test = len(dataset_test)
manifest = []

print(f"Starting to prepare manifest for {num_samples_test} samples...")

for i in range(num_samples_test):
    try:
        sample = dataset_test[i]
        raw_text = sample['transcription']

        # Save the audio file as WAV
        # Extracting audio array and sampling rate from the dataset
        audio_array = torch.tensor(sample['audio']['array']).unsqueeze(0)
        sr = sample['audio']['sampling_rate']

        # Calculate duration in Encodec tokens (VoiceCraft standard)
        # Encodec at 16kHz produces 50 tokens per second.
        # Formula: (num_samples / sampling_rate) * 50
        num_audio_samples = audio_array.shape[1]
        duration_frames = int((num_audio_samples / sr) * 50)
        
        wav_filename = f"sample_{i}.wav"
        wav_path = os.path.join(output_wav_dir, wav_filename)
        
        # Physical save to disk using torchaudio
        torchaudio.save(wav_path, audio_array, sr)
    
        # Get Phonemes
        phonemes_list = text_tokenizer_he(raw_text)[0]
        phonemes_str = " ".join(phonemes_list)
        
        # Append to manifest with relative path for VoiceCraft
        manifest.append({
            "audio_filepath": f"{output_wav_dir}/{wav_filename}",
            "text": raw_text,
            "phonemes": phonemes_str,
            "duration_frames": duration_frames
        })

        # Print sample 0 to verify everything looks correct
        if i == 0:
            print(f"--- DEBUG Sample 0 ---")
            print(f"Text: {raw_text}")
            print(f"Phonemes: {phonemes_str[:50]}...")
            print(f"Duration: {duration_frames} frames")
            print(f"----------------------")
        
        if i % 100 == 0: print(f"Processed {i} samples...")
            
    except Exception as e:
        print(f"Error in sample {i}: {e}")

# Save the manifest
with open(manifest_path, "w", encoding="utf-8") as f:
    for entry in manifest:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print("Done! Check 'hebrew_manifest_test.jsonl'")

Starting to prepare manifest for 792 samples...
--- DEBUG Sample 0 ---
Text: מרטלי השביע אתמול ועדת בחירות זמנית cep חדשה בת תשעה חברים
Phonemes: m ʁ t l i _ ʔ ʃ v i ʔ _ ʔ t m v l _ v ʔ d t _ v χ ...
Duration: 372 frames
----------------------
Processed 0 samples...
Processed 100 samples...
Processed 200 samples...
Processed 300 samples...
Processed 400 samples...
Processed 500 samples...
Processed 600 samples...
Processed 700 samples...
Done! Check 'hebrew_manifest_test.jsonl'


In [ ]:
# # Update the package list
# !sudo apt update

# # Install espeak-ng (the modern version of espeak)
# !sudo apt install espeak-ng -y

# # Verify the installation
# !espeak-ng --version

In [ ]:
# # Environment setup

# conda create -n voicecraft python=3.9.16
# conda activate voicecraft

# pip install -e git+https://github.com/facebookresearch/audiocraft.git@c5157b5bf14bf83449c17ea1eeb66c19fb4bc7f0#egg=audiocraft
# pip install xformers==0.0.22
# pip install torchaudio==2.0.2 torch==2.0.1 # this assumes your system is compatible with CUDA 11.7, otherwise checkout https://pytorch.org/get-started/previous-versions/#v201
# apt-get install ffmpeg # if you don't already have ffmpeg installed
# apt-get install espeak-ng # backend for the phonemizer installed below
# pip install tensorboard==2.16.2
# pip install phonemizer==3.2.1
# pip install datasets==2.16.0
# pip install torchmetrics==0.11.1
# pip install huggingface_hub==0.22.2
# # install MFA for getting forced-alignment, this could take a few minutes
# conda install -c conda-forge montreal-forced-aligner=2.2.17 openfst=1.8.2 kaldi=5.5.1068
# # install MFA english dictionary and model
# mfa model download dictionary english_us_arpa
# mfa model download acoustic english_us_arpa
# # pip install huggingface_hub
# # conda install pocl # above gives an warning for installing pocl, not sure if really need this

# # to run ipynb
# conda install -n voicecraft ipykernel --no-deps --force-reinstall


In [ ]:
# Installing VoiceCraft in editable mode so changes take effect immediately
# !pip install -e .